In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CSV_PATH = "generated_dataset/train.csv"
IMAGE_DIR = "generated_dataset/train"
IMG_SIZE = (64, 224)
BATCH_SIZE = 64

print("Device:", DEVICE)

Device: cpu


## Dataset

In [3]:
df = pd.read_csv(CSV_PATH)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["angle"]
)

In [4]:
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = train_transform

In [5]:
class TextOrientationDataset(Dataset):

    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(IMAGE_DIR)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        if row["angle"] == 0:
            image_path = (self.image_dir + "/0/" + row["filename"])
        else: 
            image_path = (self.image_dir + "/180/" + row["filename"])

        image = Image.open(image_path).convert("RGB")
        label = 0 if row["angle"] == 0 else 1

        if self.transform:
            image = self.transform(image)

        return image, label

In [6]:
train_dataset = TextOrientationDataset(
    train_df,
    IMAGE_DIR,
    train_transform
)

val_dataset = TextOrientationDataset(
    val_df,
    IMAGE_DIR,
    val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

Train: 16000
Val: 4000


# Модель и Обучение

In [7]:
EPOCHS = 5
LR = 1e-4

In [8]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 1)

model = model.to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=LR)

In [9]:
print(len(train_loader), train_loader.num_workers, train_loader.batch_size)

250 2 64


In [ ]:
for epoch in range(EPOCHS):

    # TRAIN
    model.train()

    train_correct = 0
    train_total = 0
    train_loss = 0

    for step, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).float().unsqueeze(1)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Вероятность класса 1
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long().squeeze(1)

        train_correct += (preds == labels.squeeze(1).long()).sum().item()
        train_total += labels.size(0)

        if step % 25 == 0:
            print(f"  step {step}/{len(train_loader)} loss={loss.item():.4f}", flush=True)

    train_acc = train_correct / train_total

    # VALIDATION
    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE).float().unsqueeze(1)

            logits = model(images)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long().squeeze(1)

            val_correct += (preds == labels.squeeze(1).long()).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch+1}: "
        f"loss={train_loss/len(train_loader):.4f}, "
        f"train_acc={train_acc:.4f}, "
        f"val_acc={val_acc:.4f}"
    )